# FigureSI_ — Ximea vs Prime95B: simulated localisation precision comparison

*Makes up Figure S44.*

A bootstrap Monte Carlo sweep over a range of photon counts, for three single-molecule dyes
(ATTO 488, ATTO 565, ATTO 647N), at a fixed background of 10 photons/pixel, predicting
achievable localisation precision (`sigma_xy`) for the Ximea (Bayer CFA) and Prime95B (mono)
cameras from first principles, using each camera's own real noise characteristics.

**Ximea**: real calibration maps from `Camera_Calibrations/Ximea_Camera/`, via
`MultiC_Sim_Funcs.test_simulation_method` with `FittingStrategy.STANDARD_DATA` — the same
strategy `SR_Functions.fit_SM_data` always uses for real single-molecule data (not the older,
purely isotropic `STANDARD`, and not `ELLIPTICAL`, which is `fit_tracking_data`'s own default
for a different purpose).

**Prime95B**: QE curve from `Spectra/Camera_QE/Prime95B.csv`, gain/read-noise from the
manufacturer datasheet (2 e-/ADU, 1.6 e- RMS — there's no calibration-map file for this
camera). `test_simulation_method` can't run a true single-channel (mono) camera (its RMSE
bookkeeping hard-assumes >= 2 colour channels), so this camera uses a bespoke bootstrap loop
built from the same lower-level `PSF_Functions` primitives, fit with `Image_Analysis_Functions.
fit_puncta_parallel_method(..., FittingStrategy.NOCOLOUR)` — the same mono fitting strategy
used for real Prime95B data elsewhere in this project (mono has no Bayer geometry to fit
elliptically against, so `NOCOLOUR` — not an elliptical variant — is the correct choice here).

**Units note**: this codebase's `gain` is ADU per electron (verified against Ximea's own
calibration: `sqrt(variance.tif) == readnoise.tif * gain.tif` pixel-for-pixel) — the
reciprocal of how manufacturers usually quote gain (`e-/ADU`), so the Prime95B
camera-parameters cell below converts accordingly.

**`FAST_MODE` flag:** `True` (default) sweeps ~8 photon levels x 150 bootstrap fits per
condition instead of the published 20 levels x 10,000 bootstrap fits — same 3 dyes and both
cameras, just coarser and noisier, in well under a minute. `FAST_MODE = False` reproduces the
published sweep exactly.

In [ ]:
import glob
import time
import types
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from scipy.spatial.distance import cdist
import xarray as xr

from pyS3M import IOFunctions, SpectralFunctions, MaskFunctions, PSFFunctions, sCMOSFunctions, PlottingBase
from pyS3M.simulation.multicolour import FittingStrategy as XimeaFittingStrategy, SimulationConfig, MultiC_Sim_Funcs
# Bare import (not `from src.ImageAnalysisFunctions import ...`): PSFFunctions/IOFunctions add
# src/ to sys.path and bare-import internally, so a `src.`-qualified import here would register
# a second, distinct FittingStrategy Enum -> spurious KeyError. Aliased to avoid confusion with
# the Bayer-only FittingStrategy above -- only used for the Prime95B (mono, NOCOLOUR) path.
from pyS3M.ImageAnalysisFunctions import FittingStrategy as MonoFittingStrategy, Image_Analysis_Functions

IO = IOFunctions.IO_Functions()
S_F = SpectralFunctions.Spectral_Funcs()
M_F = MaskFunctions.Mask_Functions()
PSF = PSFFunctions.PSF_Functions()
sCMOS = sCMOSFunctions.sCMOS_Functions()
MSF = MultiC_Sim_Funcs()
IAF_mono = Image_Analysis_Functions()
plotter = PlottingBase.PublicationPlotter()

smoothing_function = types.SimpleNamespace(
    args={"sigma": 1.5}, extent=1.5,
    smoothing_function=sCMOS.gaussian_filter_stack, data_arg="image",
)

REPO_ROOT = Path.cwd().resolve().parents[2]
CALIBRATION_DIR = REPO_ROOT / "Camera_Calibrations" / "Ximea_Camera"
OUT_ROOT = Path("outputs") / "XimeavsPrime95B"  # git-ignored
FIGURE_DIR = Path("figures")                     # git-ignored (*.svg)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
OUT_ROOT.mkdir(parents=True, exist_ok=True)

FAST_MODE = True  # False -> the published 20-level / 10,000-bootstrap sweep
print(f"FAST_MODE={FAST_MODE}")

## Shared sweep parameters

In [ ]:
dyes = ["ATTO 488", "ATTO 565", "ATTO 647N"]
NA = 1.49
BACKGROUND_PHOTONS = 10.0  # fixed, not swept
N_BOOTSTRAP = 150 if FAST_MODE else 10_000
n_photon_steps = 8 if FAST_MODE else 20

# Single-molecule-relevant photon range (not the bright-bead range used in the real
# Ximea-vs-Prime95B experimental data).
n_photon_space = np.unique(np.round(np.geomspace(200, 20000, n_photon_steps)))

print(f"Dyes: {dyes}")
print(f"Photon levels: {len(n_photon_space)}  ({n_photon_space[0]:.0f}-{n_photon_space[-1]:.0f})")
print(f"n_bootstrap={N_BOOTSTRAP}  background={BACKGROUND_PHOTONS} photons/pixel")

## Ximea camera parameters (real calibration maps) & sweep

In [ ]:
gain = IO.read_tiff(str(CALIBRATION_DIR / "gain.tif"))
offset = IO.read_tiff(str(CALIBRATION_DIR / "offset.tif"))
variance = IO.read_tiff(str(CALIBRATION_DIR / "variance.tif"))
readnoise = float(np.median(IO.read_tiff(str(CALIBRATION_DIR / "readnoise.tif"))))
rqe = IO.read_tiff(str(CALIBRATION_DIR / "rqe.tif"))
print(f"Ximea calibration loaded -- sensor: {gain.shape}, readnoise: {readnoise:.2f} e-")

R_base, G_base, B_base, wavelength = S_F.getpixelefficiency()
pixel_QYs = np.vstack([B_base, G_base, R_base])  # B=0, G=1, R=2
mosaic = M_F.mosaic_unit

pixel_order = ["B", "G", "R"]
pixel_order_indices = {"B": 0, "G": 1, "R": 2}

PIXEL_SIZE_XIMEA_NM = 69.0
filters = []
image_size = 14  # seed size; test_simulation_method resizes via mosaic_unit

ximea_camera_params = {
    "gain": np.full((image_size, image_size), np.median(gain)),
    "offset": np.full((image_size, image_size), np.median(offset)),
    "variance": np.full((image_size, image_size), np.median(variance)),
    "readnoise": readnoise,
    "rqe": np.full((image_size, image_size), np.median(rqe)),
    "masks": M_F.get_masks(size_x=image_size, size_y=image_size),
    "pixel_QYs": pixel_QYs,
    "pixel_order": pixel_order,
    "pixel_order_indices": pixel_order_indices,
    "mosaic_unit": mosaic,
}
print(f"Ximea camera params ready -- base image size: {image_size}x{image_size} px")

In [ ]:
ximea_config = SimulationConfig(
    n_bootstrap=N_BOOTSTRAP, background_photons=BACKGROUND_PHOTONS, NA=NA, pixel_size=PIXEL_SIZE_XIMEA_NM,
    save_raw_results=True, subtractx0y0=False, saverawimages=False, verbose=False,
    use_stochastic_photons=True, n_unit_cells=7,
)

XIMEA_FLAG = "ximea_"
t_start = time.time()
for i_dye, dye in enumerate(dyes):
    print(f"[{i_dye + 1}/{len(dyes)}] {dye}", end="  ", flush=True)
    MSF.test_simulation_method(
        dye=dye, filters=filters, wavelength=wavelength, camera_parameters=ximea_camera_params,
        save_folder=str(OUT_ROOT), n_photon_space=n_photon_space, smoothing_function=smoothing_function,
        strategy=XimeaFittingStrategy.STANDARD_DATA, starting_flag=XIMEA_FLAG, config=ximea_config, overwrite=True,
    )
    print(f"done ({(time.time() - t_start) / 60:.1f} min elapsed)")

print(f"\nXimea sweep complete in {(time.time() - t_start) / 60:.1f} min.")

## Ximea analysis: load raw HDF5 results + ground truth, compute `sigma_xy`

In [ ]:
metrics = ["sigma_xy", "sigma_colour"]
a_cols = [f"A_{l}" for l in pixel_order]
n_unit = ximea_config.n_unit_cells

XIMEA_XR = xr.DataArray(
    data=np.full([len(dyes), len(n_photon_space), len(metrics)], np.nan),
    coords={"dye": dyes, "n_photons": n_photon_space, "metric": metrics},
    dims=["dye", "n_photons", "metric"],
)

missing = []
for i_d, dye in enumerate(dyes):
    dye_str = dye.replace("/", "-")

    h5_path = OUT_ROOT / f"{XIMEA_FLAG}LM_method_{dye_str}_rawresults.h5"
    gt_path = sorted(glob.glob(str(OUT_ROOT / f"{XIMEA_FLAG}*{dye_str}*groundtruth*.csv")))
    ip_path = sorted(glob.glob(str(OUT_ROOT / f"{XIMEA_FLAG}*{dye_str}*input_parameters*.csv")))

    if not h5_path.exists() or not gt_path or not ip_path:
        missing.append(dye)
        continue

    gt = pd.read_csv(gt_path[0])
    gt_px = np.vstack([gt["x0"].to_numpy() / PIXEL_SIZE_XIMEA_NM, gt["y0"].to_numpy() / PIXEL_SIZE_XIMEA_NM]).T

    ip_row = pd.read_csv(ip_path[0]).to_numpy()[0]
    colour_raw = ip_row[-3:]
    colour_true = np.expand_dims(colour_raw / colour_raw.sum(), 0)

    df_raw = pd.read_hdf(str(h5_path), key="data")

    for k in range(len(n_photon_space)):
        sub = df_raw[df_raw["photon_level"] == k]
        filt = (
            (sub["xc"].to_numpy() > 0) & (sub["yc"].to_numpy() > 0)
            & (sub["s_x"].to_numpy() > 0) & (sub["s_y"].to_numpy() > 0)
            & (sub["xc"].to_numpy() < n_unit * mosaic.shape[0]) & (sub["yc"].to_numpy() < n_unit * mosaic.shape[1])
        )
        if filt.sum() < 5:
            continue

        ex = (sub["xc"].to_numpy()[filt] - gt_px[filt, 0]) * PIXEL_SIZE_XIMEA_NM
        ey = (sub["yc"].to_numpy()[filt] - gt_px[filt, 1]) * PIXEL_SIZE_XIMEA_NM
        XIMEA_XR[i_d, k, 0] = np.sqrt((np.nanstd(ex) ** 2 + np.nanstd(ey) ** 2) / 2)

        colours = np.vstack([sub[c].to_numpy()[filt] for c in a_cols]).T
        XIMEA_XR[i_d, k, 1] = float(np.nanstd(cdist(colours, colour_true)))

if missing:
    print(f"Missing: {missing}")
XIMEA_XR.to_netcdf(str(OUT_ROOT / "XIMEA_XR.nc"))
print(f"DataArray shape: {XIMEA_XR.shape}  dims: {list(XIMEA_XR.dims)}")
print(f"sigma_xy range: {float(np.nanmin(XIMEA_XR.sel(metric='sigma_xy'))):.1f}-{float(np.nanmax(XIMEA_XR.sel(metric='sigma_xy'))):.1f} nm")

## Prime95B camera parameters

QE curve from `Spectra/Camera_QE/Prime95B.csv` (two columns, no header,
`(wavelength_nm, QE_percent)`), interpolated onto the same wavelength grid Ximea uses.
Gain/read-noise/offset come from the manufacturer datasheet, not a measured calibration file
(none exists for this camera).

In [ ]:
p95b_raw = np.loadtxt(REPO_ROOT / "Spectra" / "Camera_QE" / "Prime95B.csv", delimiter=",")
prime95b_qe = np.interp(wavelength, p95b_raw[:, 0], p95b_raw[:, 1] / 100.0)

PRIME95B_GAIN_ADU_PER_E = 1.0 / 2.0  # 0.5 ADU/e-, from the "2 e-/ADU" spec
PRIME95B_READNOISE_E = 1.6           # electrons RMS, direct
PRIME95B_VARIANCE_ADU2 = (PRIME95B_READNOISE_E * PRIME95B_GAIN_ADU_PER_E) ** 2
PRIME95B_OFFSET_ADU = 100.0          # fixed default (no calibration file)
PIXEL_SIZE_PRIME95B_NM = 69.0        # fixed default
ROI_SIZE_PRIME95B = 14

print(f"Prime95B QE range: {prime95b_qe.min():.3f}-{prime95b_qe.max():.3f}")
print(f"Prime95B gain={PRIME95B_GAIN_ADU_PER_E} ADU/e-  readnoise={PRIME95B_READNOISE_E} e-  "
      f"variance={PRIME95B_VARIANCE_ADU2:.3f} ADU^2  offset={PRIME95B_OFFSET_ADU} ADU")

## Prime95B bootstrap precision loop (bespoke — see the intro cell for why)

In [ ]:
from pyS3M.simulation.vectorial_sampler import photon_by_photon_photoelectrons, radial_dipole_psf

def simulate_prime95b_sigma_xy(dye, n_photons, n_bootstrap, roi_size=ROI_SIZE_PRIME95B,
                                gain=PRIME95B_GAIN_ADU_PER_E, variance=PRIME95B_VARIANCE_ADU2):
    '''Bootstrap sigma_xy (nm) for one dye/photon level on a mono camera using the Prime95B QE
    curve. gain/variance default to the Prime95B manufacturer spec -- pass Ximea's own measured
    gain/variance instead to isolate the pixel-QE + spatial-patterning contribution from the
    electronics contribution (see the hybrid-camera cells below).'''
    avg_wl, abs_qe = S_F.get_pixel_fractions_dye_and_filters(
        dyes=[dye], filters=[], wavelength=wavelength, pixel_QYs=prime95b_qe[np.newaxis, :], normalized=False,
    )
    abs_qe = float(abs_qe)
    x0 = y0 = roi_size / 2.0

    gain_map = np.full((roi_size, roi_size), gain)
    offset_map = np.full((roi_size, roi_size), PRIME95B_OFFSET_ADU)
    variance_map = np.full((roi_size, roi_size), variance)

    # Signal: the simulator's default model (as the Ximea sweep) -- photon by photon with the
    # vectorial dipole PSF, each photon with its own wavelength from the dye's spectrum and
    # detected with the Prime95B QE at that wavelength; n_photons land in the ROI.
    rng = np.random.default_rng(np.random.randint(0, 2**31 - 1, size=4))
    signal_pe = photon_by_photon_photoelectrons(
        radial_dipole_psf(NA), MultiC_Sim_Funcs.filtered_emission_spectrum(dye, [], wavelength), wavelength,
        prime95b_qe[np.newaxis, :], np.zeros((roi_size, roi_size), dtype=int), n_photons,
        np.full(n_bootstrap, x0), np.full(n_bootstrap, y0), PIXEL_SIZE_PRIME95B_NM, rng,
    )
    # Background is injected pre-QE, then binomially thinned by abs_qe -- pre-dividing by abs_qe
    # cancels that thinning, matching the Ximea sweep's convention (background defined in
    # post-QE photoelectrons).
    background_pe = rng.binomial(rng.poisson(BACKGROUND_PHOTONS / abs_qe, signal_pe.shape), abs_qe)

    puncta, smoothed, weights, coords, planes = [], [], [], [], []
    for i in range(n_bootstrap):
        photoelectrons = signal_pe[i] + background_pe[i]
        raw_roi = PSF.photoelectrons_to_image(photoelectrons.astype(np.float64), gain_map, offset_map, variance_map)
        smoothed_roi = gaussian_filter(raw_roi, sigma=1.5)
        weight_roi = 1.0 / np.clip(variance_map + gain_map * np.clip(raw_roi - offset_map, 0, None), 1e-6, None)

        puncta.append(raw_roi)
        smoothed.append(smoothed_roi)
        weights.append(weight_roi)
        coords.append([0.0, 0.0])
        planes.append(i)

    fit_arr, _ = IAF_mono.fit_puncta_parallel_method(puncta, smoothed, weights, coords, planes, MonoFittingStrategy.NOCOLOUR)
    xc, yc = fit_arr[:, 0], fit_arr[:, 1]
    valid = (xc > 0) & (yc > 0) & (fit_arr[:, 2] > 0) & (fit_arr[:, 3] > 0) & (xc < roi_size) & (yc < roi_size)
    if valid.sum() < 5:
        return np.nan, int(valid.sum())

    ex = (xc[valid] - x0) * PIXEL_SIZE_PRIME95B_NM
    ey = (yc[valid] - y0) * PIXEL_SIZE_PRIME95B_NM
    sigma_xy = float(np.sqrt((np.nanstd(ex) ** 2 + np.nanstd(ey) ** 2) / 2))
    return sigma_xy, int(valid.sum())

In [ ]:
PRIME95B_XR = xr.DataArray(
    data=np.full([len(dyes), len(n_photon_space)], np.nan),
    coords={"dye": dyes, "n_photons": n_photon_space}, dims=["dye", "n_photons"],
)

t_start = time.time()
n_total = len(dyes) * len(n_photon_space)
i_run = 0
for i_d, dye in enumerate(dyes):
    for i_p, n_photons in enumerate(n_photon_space):
        i_run += 1
        sigma_xy, n_valid = simulate_prime95b_sigma_xy(dye, float(n_photons), N_BOOTSTRAP)
        PRIME95B_XR[i_d, i_p] = sigma_xy
        if i_run % max(1, n_total // 5) == 0 or i_run == n_total:
            elapsed = (time.time() - t_start) / 60
            print(f"[{i_run}/{n_total}] {dye}  N={n_photons:.0f}  sigma_xy={sigma_xy:.2f} nm  valid={n_valid}/{N_BOOTSTRAP}  ({elapsed:.1f} min elapsed)")

PRIME95B_XR.to_netcdf(str(OUT_ROOT / "PRIME95B_XR.nc"))
print(f"\nPrime95B sweep complete in {(time.time() - t_start) / 60:.1f} min.")

## Figure: Ximea vs Prime95B — localisation precision vs photon count, per dye

In [ ]:
fig, axs = plotter.two_column_plot(ncols=len(dyes), width_ratios=list(np.ones(len(dyes))), width=6.69, height=2.6)
for i_d, dye in enumerate(dyes):
    ax = axs[i_d]
    ximea_sigma = XIMEA_XR.isel(dye=i_d).sel(metric="sigma_xy").values
    prime_sigma = PRIME95B_XR.isel(dye=i_d).values

    ax = plotter.line_plot(ax, n_photon_space, ximea_sigma, color="black", label="Ximea")
    ax = plotter.line_plot(ax, n_photon_space, prime_sigma, color="red", label="Prime95B")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel(r"$N_{\rm photons}$")
    ax.set_xlim([n_photon_space.min(), n_photon_space.max()])
    ax.set_ylim([0.5, 80])
    ax.set_ylabel(r"$\sigma_{xy}$ / nm" if i_d == 0 else "")
    ax.set_title(dye, fontsize=8)
    if i_d == len(dyes) - 1:
        ax.legend(fontsize=7, frameon=False)

plotter.save_or_show(fig, save_path=str(FIGURE_DIR / "ximea_vs_prime95b_precision.svg"), show=True)

## Figure: precision-ratio vs QE-predicted improvement

The dashed red line is the improvement predicted purely from the QE ratio
(`sqrt(prime95B_QE / mean_bayer_QE)`, no simulation) — how closely the simulated
`sigma_xy` ratio tracks it shows how much of the Ximea-vs-Prime95B gap is explained by raw
photon-collection efficiency alone (vs. e.g. Bayer demosaicing/colour-fit overhead).

In [ ]:
prime95B_QEs = S_F.get_absolute_pixel_QYs(dyes=dyes, filters=[], wavelength=wavelength, pixel_QYs=prime95b_qe)[1]
bayer_QEs = S_F.get_absolute_pixel_QYs(dyes=dyes, filters=[], wavelength=wavelength, pixel_QYs=pixel_QYs)[1]

B_QE, G_QE, R_QE = bayer_QEs[:, 0], bayer_QEs[:, 1], bayer_QEs[:, 2]
overall_QE = np.vstack([B_QE, G_QE, G_QE, R_QE])  # G counted twice -- BGGR Bayer unit-cell weighting
bayer_QEs_overall = np.mean(overall_QE, axis=0)

improvement_level = np.sqrt(prime95B_QEs / bayer_QEs_overall)
print("QE-predicted sigma_xy(Ximea)/sigma_xy(Prime95B) per dye:", dict(zip(dyes, improvement_level.round(2))))

fig, axs = plotter.two_column_plot(ncols=len(dyes), width_ratios=list(np.ones(len(dyes))), width=6.69, height=2.6)
for i_d, dye in enumerate(dyes):
    ax = axs[i_d]
    ximea_sigma = XIMEA_XR.isel(dye=i_d).sel(metric="sigma_xy").values
    prime_sigma = PRIME95B_XR.isel(dye=i_d).values

    ax = plotter.line_plot(ax, n_photon_space, ximea_sigma / prime_sigma, color="black")
    ax.set_xscale("log")
    ax.set_xlabel(r"$N_{\rm photons}$")
    ax.set_xlim([n_photon_space.min(), n_photon_space.max()])
    ax.set_ylim([0, 3.5])
    ax.hlines(xmin=n_photon_space.min(), xmax=n_photon_space.max(), y=improvement_level[i_d], ls="--", color="red")
    ax.set_ylabel(r"$\sigma_\mathrm{xy (Ximea)}$/$\sigma_\mathrm{xy (Prime 95B)}$" if i_d == 0 else "")
    ax.set_title(dye, fontsize=8)

plotter.save_or_show(fig, save_path=str(FIGURE_DIR / "ximea_vs_prime95b_improvement.svg"), show=True)

## Hybrid control: Prime95B QE, Ximea electronics

Isolates the pixel-QE + spatial-patterning (mono vs Bayer) contribution to the precision
difference from the electronics (gain/read-noise) contribution: same Prime95B QE curve and
mono (no Bayer) geometry as the Prime95B sweep above, but gain/variance taken directly from
Ximea's own measured calibration maps, not the Prime95B manufacturer spec.

In [ ]:
HYBRID_GAIN_ADU_PER_E = float(np.median(gain))      # Ximea's measured gain
HYBRID_VARIANCE_ADU2 = float(np.median(variance))    # Ximea's measured (read-noise) variance

print(f"Hybrid (Prime95B QE + Ximea electronics): gain={HYBRID_GAIN_ADU_PER_E:.3f} ADU/e-  "
      f"variance={HYBRID_VARIANCE_ADU2:.3f} ADU^2  (readnoise={readnoise:.2f} e-, same as Ximea)")

HYBRID_XR = xr.DataArray(
    data=np.full([len(dyes), len(n_photon_space)], np.nan),
    coords={"dye": dyes, "n_photons": n_photon_space}, dims=["dye", "n_photons"],
)

t_start = time.time()
i_run = 0
for i_d, dye in enumerate(dyes):
    for i_p, n_photons in enumerate(n_photon_space):
        i_run += 1
        sigma_xy, n_valid = simulate_prime95b_sigma_xy(
            dye, float(n_photons), N_BOOTSTRAP, gain=HYBRID_GAIN_ADU_PER_E, variance=HYBRID_VARIANCE_ADU2,
        )
        HYBRID_XR[i_d, i_p] = sigma_xy
        if i_run % max(1, n_total // 5) == 0 or i_run == n_total:
            elapsed = (time.time() - t_start) / 60
            print(f"[{i_run}/{n_total}] {dye}  N={n_photons:.0f}  sigma_xy={sigma_xy:.2f} nm  valid={n_valid}/{N_BOOTSTRAP}  ({elapsed:.1f} min elapsed)")

HYBRID_XR.to_netcdf(str(OUT_ROOT / "HYBRID_XR.nc"))
print(f"\nHybrid sweep complete in {(time.time() - t_start) / 60:.1f} min.")

## Figure: Ximea vs Prime95B vs Hybrid

In [ ]:
fig, axs = plotter.two_column_plot(ncols=len(dyes), width_ratios=list(np.ones(len(dyes))), width=6.69, height=2.6)
for i_d, dye in enumerate(dyes):
    ax = axs[i_d]
    ximea_sigma = XIMEA_XR.isel(dye=i_d).sel(metric="sigma_xy").values
    prime_sigma = PRIME95B_XR.isel(dye=i_d).values
    hybrid_sigma = HYBRID_XR.isel(dye=i_d).values

    ax = plotter.line_plot(ax, n_photon_space, ximea_sigma, color="black", label="Ximea")
    ax = plotter.line_plot(ax, n_photon_space, prime_sigma, color="red", label="Prime95B")
    ax = plotter.line_plot(ax, n_photon_space, hybrid_sigma, color="tab:blue", label="Prime95B QE + Ximea electronics")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xlabel(r"$N_{\rm photons}$")
    ax.set_xlim([n_photon_space.min(), n_photon_space.max()])
    ax.set_ylabel(r"$\sigma_{xy}$ / nm" if i_d == 0 else "")
    ax.set_title(dye, fontsize=8)
    if i_d == len(dyes) - 1:
        ax.legend(fontsize=6, frameon=False)

plotter.save_or_show(fig, save_path=str(FIGURE_DIR / "ximea_vs_prime95b_hybrid.svg"), show=True)

In [ ]:
fig, axs = plotter.two_column_plot(ncols=len(dyes), width_ratios=list(np.ones(len(dyes))), width=6.69, height=2.6)
for i_d, dye in enumerate(dyes):
    ax = axs[i_d]
    ximea_sigma = XIMEA_XR.isel(dye=i_d).sel(metric="sigma_xy").values
    hybrid_sigma = HYBRID_XR.isel(dye=i_d).values

    ax = plotter.line_plot(ax, n_photon_space, ximea_sigma / hybrid_sigma, color="black")
    ax.set_xscale("log")
    ax.set_xlabel(r"$N_{\rm photons}$")
    ax.set_xlim([n_photon_space.min(), n_photon_space.max()])
    ax.hlines(xmin=n_photon_space.min(), xmax=n_photon_space.max(), y=improvement_level[i_d], ls="--", color="red")
    ax.set_ylabel(r"$\sigma_\mathrm{xy (Ximea)}$/$\sigma_\mathrm{xy (Prime 95B, hybrid)}$" if i_d == 0 else "")
    ax.set_title(dye, fontsize=8)

plotter.save_or_show(fig, save_path=str(FIGURE_DIR / "ximea_vs_fakeprime95b_improvement.svg"), show=True)

## Next steps

Set `FAST_MODE = False` above and re-run for the published sweep (20 photon levels x 10,000
bootstrap fits x 3 dyes x 3 camera configurations).